In [ ]:
import torch


class Rotary(torch.nn.Module):
    def __init__(self, dim, base=10000):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)
        self.seq_len_cached = None
        self.cos_cached = None
        self.sin_cached = None

    def forward(self, x, seq_dim=1):
        seq_len = x.shape[seq_dim]
        if seq_len != self.seq_len_cached:
            self.seq_len_cached = seq_len
            t = torch.arange(x.shape[seq_dim], device=x.device).type_as(self.inv_freq)
            freqs = torch.einsum("i,j->ij", t, self.inv_freq)
            emb = torch.cat((freqs, freqs), dim=-1).to(x.device)
            self.cos_cached = emb.cos()[:, None, None, :]
            self.sin_cached = emb.sin()[:, None, None, :]
        return self.cos_cached, self.sin_cached


# rotary pos emb helpers:

def rotate_half(x):
    x1, x2 = x[..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2 :]
    return torch.cat(
        (-x2, x1), dim=x1.ndim - 1
    )  # dim=-1 triggers a bug in torch < 1.8.0


@torch.jit.script
def apply_rotary_pos_emb(q, k, cos, sin):
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)

# here dimensions of q,k are: 
# of GPT-NeoX, following Megatron, is [seq, batch, heads, hdim]

In [ ]:
# roformer paper github
# how do we get sinusodial_pos though?
def apply_rotary(x, sinusoidal_pos=None):
    # original dimensions of x is: (B,T,d)
    if sinusoidal_pos is None:
        return x
    sin, cos = sinusoidal_pos # what dim is sin and cos? is it a array -> must match d/2, or a single value?
    # we say that the model does m_theta based on position
    # x.shape [batch, seq_len, 2]
    x1, x2 = x[..., 0::2], x[..., 1::2] # here x1 is say (B, T, d/2)
    # sin and cos of size: (T, d/2)
    # [cos_nθ, -sin_nθ] [x1]
    # [sin_nθ,  cos_nθ] [x2]
    # => [x1 * cos_nθ - x2 * sin_nθ, x1 * sin_nθ + x2 * cos_nθ]
    # 苏神的rotary，使用了下面的计算方法。
    # return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2, -1)
    # 考虑到矩阵乘法torch.einsum("bhmd,bhnd->bhmn", q, k)，因此可以直接在最后一个维度拼接（无需奇偶交错）
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1) # (B,T, d/2 *2) => (B,T,d) what size does this become?



In [14]:
import torch
x = torch.tensor([[[1,2,3,4],[2,4,7,3]],[[2,6,8,1],[3,5,6,4]],[[4,7,6,6],[6,5,1,1]],[[5,4,3,3],[3,2,2,4]]])
print(x)

tensor([[[1, 2, 3, 4],
         [2, 4, 7, 3]],

        [[2, 6, 8, 1],
         [3, 5, 6, 4]],

        [[4, 7, 6, 6],
         [6, 5, 1, 1]],

        [[5, 4, 3, 3],
         [3, 2, 2, 4]]])


In [15]:
x.shape # torch.Size([4, 2, 2]) -> (batch, seq_len, 2) -> each token 2 dimensions

torch.Size([4, 2, 4])

In [29]:
x1, x2 = x[..., 0::2], x[..., 1::2]
print(x1)
t1 = torch.cat([x1  - x2 ], dim=-1)
t2 = torch.cat([x1  + x2 ], dim=-1)
print(t1)
print(t2)

tensor([[[1, 3],
         [2, 7]],

        [[2, 8],
         [3, 6]],

        [[4, 6],
         [6, 1]],

        [[5, 3],
         [3, 2]]])
tensor([[[-1, -1],
         [-2,  4]],

        [[-4,  7],
         [-2,  2]],

        [[-3,  0],
         [ 1,  0]],

        [[ 1,  0],
         [ 1, -2]]])
tensor([[[ 3,  7],
         [ 6, 10]],

        [[ 8,  9],
         [ 8, 10]],

        [[11, 12],
         [11,  2]],

        [[ 9,  6],
         [ 5,  6]]])


In [17]:
t = torch.tensor([1,2,3,4])

(tensor([1, 3]), tensor([2, 4]))

In [ ]:
# [sequence_length, embed_size_per_head] -> sin & cos [batch_size, num_heads, sequence_length, embed_size_per_head // 2]
sinusoidal_pos = self.embed_positions(hidden_states.shape[1], past_key_values_length)[
    None, None, :, :
].chunk(2, dim=-1)

In [ ]:
class RoFormerSinusoidalPositionalEmbedding(nn.Embedding):
    """This module produces sinusoidal positional embeddings of any length."""

    def __init__(
        self, num_positions: int, embedding_dim: int, padding_idx: Optional[int] = None
    ):
        super().__init__(num_positions, embedding_dim)
        self.weight = self._init_weight(self.weight)

    @staticmethod
    def _init_weight(out: nn.Parameter):
        """
        Identical to the XLM create_sinusoidal_embeddings except features are not interleaved. The cos features are in
        the 2nd half of the vector. [dim // 2:]
        """
        n_pos, dim = out.shape
        position_enc = np.array(
            [
                [pos / np.power(10000, 2 * (j // 2) / dim) for j in range(dim)]
                for pos in range(n_pos)
            ]
        )
        out.requires_grad = False  # set early to avoid an error in pytorch-1.8+
        sentinel = dim // 2 if dim % 2 == 0 else (dim // 2) + 1
        out[:, 0:sentinel] = torch.FloatTensor(np.sin(position_enc[:, 0::2])) # taking sin values of 0,2,4,6 ...
        out[:, sentinel:] = torch.FloatTensor(np.cos(position_enc[:, 1::2])) # taking cos values of 1,3,5,7 ...
        out.detach_()
        return out # out of shape: seq_len, n_embd, but n_embd/2 contain sin and after contain cos

    @torch.no_grad()
    def forward(self, seq_len: int, past_key_values_length: int = 0):
        """`input_ids_shape` is expected to be [bsz x seqlen]."""
        positions = torch.arange(
            past_key_values_length,
            past_key_values_length + seq_len,
            dtype=torch.long,
            device=self.weight.device,
        )
        return super().forward(positions)

In [ ]:
self.embed_positions = RoFormerSinusoidalPositionalEmbedding(
    config.max_position_embeddings,
    config.hidden_size // config.num_attention_heads,
)

sinusoidal_pos = self.embed_positions(hidden_states.shape[1], past_key_values_length)[
    None, None, :, :
].chunk(2, dim=-1)  
# here we chunk the data based on the last dim i.e n_embd into two, first part contains sin, second cos

# then we pass on this sinusodial_pos

In [ ]:
# so for each position of token, the rotation is of order m*theta, where m is the token's position
# and for each embedding values withing the token's embedding, we apply pairwise rotations, but they are not same, we vary them
# we vary them by 1/(1000^(2*(i-1)/d)), where i is {1,2...d/2} and d is number of dimensions
# so say for first pair we have => m=1, theta = 1 value = 1, for second pair, m=1, theta = say d=4, 1/1000^1/4 = 0.1777
# so for second example => m=2, theta = 1, value = 2, for second pair: 2*0.1777 = 0.3554
# m=3 , theta = 1 value = 3, second pair is by orders of 0.1777*3
# m=3 , theta = 1 value = 4, second pair is by orders of 0.1777*4
# so all this is added to the keys and values of the dot product

In [ ]:
# implementing RoPE step by step
# to implement RoPE we would need to 2 things:
# the x -> input
# and the sinusodial transformation values -> theta
# theta = θi = 10000−2i/d

# rotary is applied to key and queries values

def apply_rotary(x):
    """
    # here input x is of shape: (B,T,C) => C is say n_embd
    We will splitting the C into pairs of 2
    formula:
    (
        cosm@   -sinm@
        sinm@   cosm@w
    )
    multiplied by (p1, p2)
    -> this results in -> ((p1*cosm@-p2*sinm@) =>real part, (p1*sinm@ + p2*cosm@) =>img part)
    we can use something like p1 = x[... , 0::2], p2 = x[..., 1::2], both of shapes: (B,T,C/2)
    # now we need the cos and sin parts
    # cos and sin parts will be of formula: 
    # so m@ is: m*theta where m is the token's position and theta = 10000^(-2*i/d)
    """


    return

In [8]:
import torch
d = 32
# i = torch.tensor([1,2,3,4,5,6,7])
i = torch.arange(0, 16)
theta = 10000**(-2*(i-1)/d)

In [9]:
theta

tensor([1.7783e+00, 1.0000e+00, 5.6234e-01, 3.1623e-01, 1.7783e-01, 1.0000e-01,
        5.6234e-02, 3.1623e-02, 1.7783e-02, 1.0000e-02, 5.6234e-03, 3.1623e-03,
        1.7783e-03, 1.0000e-03, 5.6234e-04, 3.1623e-04])

In [ ]:
# so we need to geometrically space the theta from 1 to 1/10000 in d/2 steps startng from start=1
# so theta becomes = 1/10000**(2*(i-1)/d)
# so for say n_embd, i will range from 1 to d/2
# so theta becomes of size d/2 => n_embd/2
# m will range from pos 1 to seq_len => T
# so a m@ will be of size (T, n_embd/2)
# we will also need to apply cos and sin to the above

# so we have n_embd split into pairs of two -> d/2
# now each individual pair will be using same value of @i i.e 1/(10000^())
def get_sin_cos(d):
    n_pairs = d/2
    theta_i_values = torch.tensor([10000**(-2*(i-1)/d) for i in range n_pairs])
    theta_values = torch.tensor()
    return

In [ ]:
d=32
theta_i_values = torch.tensor([10000**(-2*(i-1)/d) for i in range(1, d/2)], dtype=float)

TypeError: 'float' object cannot be interpreted as an integer

In [27]:
d = 32
x = torch.tensor([10000**(-2*(x-1)/d) for x in range(1, d)])
x

tensor([1.0000e+00, 5.6234e-01, 3.1623e-01, 1.7783e-01, 1.0000e-01, 5.6234e-02,
        3.1623e-02, 1.7783e-02, 1.0000e-02, 5.6234e-03, 3.1623e-03, 1.7783e-03,
        1.0000e-03, 5.6234e-04, 3.1623e-04, 1.7783e-04, 1.0000e-04, 5.6234e-05,
        3.1623e-05, 1.7783e-05, 1.0000e-05, 5.6234e-06, 3.1623e-06, 1.7783e-06,
        1.0000e-06, 5.6234e-07, 3.1623e-07, 1.7783e-07, 1.0000e-07, 5.6234e-08,
        3.1623e-08])